# 02 — Segmentación de frames y clips (Fase 0)

**Objetivo:** Generar frames espaciados (análisis estático) y clips temporales (acciones / V-JEPA).

| Entrada | Salida |
|---------|--------|
| `outputs/01_capture/` o video | `outputs/02_segments/frames/`, `clips/clip_XXXX/`, `manifest.json` |


## Prerrequisitos

Notebook **01** o un archivo `.mp4` en `INPUT_VIDEO`.


## 1. Setup


In [2]:
from __future__ import annotations

import shutil
from pathlib import Path

import cv2

from _common.io import (
    ensure_scripts_on_path,
    find_first_mp4,
    read_json,
    repo_root,
    setup_logging,
    stage_output_dir,
    write_json,
)
from loguru import logger

ensure_scripts_on_path()
setup_logging()


## 2. Configuration


In [3]:
CAPTURE_DIR = stage_output_dir("01_capture")
OUT_DIR = stage_output_dir("02_segments")
FRAMES_DIR = OUT_DIR / "frames"
CLIPS_DIR = OUT_DIR / "clips"

INPUT_VIDEO = None  # None = metadata de 01 o primer InHARD
STATIC_FPS = 1.0
CLIP_SEC = 2.0
CLIP_FRAMES_MAX = 64


## 3. Resolver fuente de video


In [4]:
meta_path = CAPTURE_DIR / "metadata.json"
if INPUT_VIDEO:
    video_path = Path(INPUT_VIDEO)
elif meta_path.is_file():
    meta = read_json(meta_path)
    video_path = Path(meta["source"])
else:
    found = find_first_mp4()
    if found is None:
        raise FileNotFoundError("No hay video. Ejecute 01 o extraiga InHARD.")
    video_path = found

if not video_path.is_file():
    raise FileNotFoundError(f"Video no encontrado: {video_path}")

logger.info("Segmentando: {}", video_path)


22:04:25 | INFO | Segmentando: /Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/data_sample/InHARD-master/01-InHARD/Segmented/RGBSegmented/Assemble system/P01_R01_0013.84_0018.88.mp4


## 4. Frames espaciados + clips


In [5]:
FRAMES_DIR.mkdir(parents=True, exist_ok=True)
CLIPS_DIR.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
static_interval = max(1, int(round(fps / max(STATIC_FPS, 0.1))))
clip_len = min(CLIP_FRAMES_MAX, max(8, int(CLIP_SEC * fps)))

static_saved = 0
clip_idx = 0
frame_i = 0
clip_frames: list = []
manifest_clips: list[dict] = []

while True:
    ok, frame = cap.read()
    if not ok:
        break
    if frame_i % static_interval == 0:
        cv2.imwrite(str(FRAMES_DIR / f"static_{static_saved:06d}.jpg"), frame)
        static_saved += 1
    clip_frames.append(frame)
    if len(clip_frames) >= clip_len:
        clip_dir = CLIPS_DIR / f"clip_{clip_idx:04d}"
        clip_dir.mkdir(parents=True, exist_ok=True)
        start_sec = (frame_i - clip_len + 1) / fps
        end_sec = frame_i / fps
        for j, cf in enumerate(clip_frames):
            cv2.imwrite(str(clip_dir / f"frame_{j:06d}.jpg"), cf)
        manifest_clips.append({
            "clip_id": f"clip_{clip_idx:04d}",
            "path": str(clip_dir.relative_to(repo_root())),
            "start_sec": round(start_sec, 3),
            "end_sec": round(end_sec, 3),
            "num_frames": len(clip_frames),
            "parent_video": str(video_path),
        })
        clip_idx += 1
        clip_frames = []
    frame_i += 1

cap.release()
logger.info("Static frames: {}, clips: {}", static_saved, clip_idx)


22:04:29 | INFO | Static frames: 6, clips: 2


## 5. Manifest


In [6]:
manifest = {
    "parent_video": str(video_path),
    "static_frames_dir": str(FRAMES_DIR.relative_to(repo_root())),
    "static_frame_count": static_saved,
    "clips": manifest_clips,
}
write_json(OUT_DIR / "manifest.json", manifest)
manifest


{'parent_video': '/Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/data_sample/InHARD-master/01-InHARD/Segmented/RGBSegmented/Assemble system/P01_R01_0013.84_0018.88.mp4',
 'static_frames_dir': 'outputs/02_segments/frames',
 'static_frame_count': 6,
 'clips': [{'clip_id': 'clip_0000',
   'path': 'outputs/02_segments/clips/clip_0000',
   'start_sec': 0.0,
   'end_sec': 1.966,
   'num_frames': 58,
   'parent_video': '/Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/data_sample/InHARD-master/01-InHARD/Segmented/RGBSegmented/Assemble system/P01_R01_0013.84_0018.88.mp4'},
  {'clip_id': 'clip_0001',
   'path': 'outputs/02_segments/clips/clip_0001',
   'start_sec': 2.0,
   'end_sec': 3.966,
   'num_frames': 58,
   'parent_video': '/Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/data_sample/InHARD-master/01-InHARD/Segmented/RGBSegmented/As

## 6. Validación


In [7]:
assert static_saved > 0 and clip_idx > 0, "Segmentación vacía"
print(f"OK — {static_saved} frames estáticos, {clip_idx} clips")


OK — 6 frames estáticos, 2 clips


## 6. Siguiente paso

**[03_detect_and_track.ipynb](03_detect_and_track.ipynb)**
